# 01 Dataset Audit - J03WPY

Notebook ini memastikan event dan tracking IDSSE dapat dibaca, memeriksa schema utama, kelengkapan possession, serta urutan timestamp. Pergantian `team_id` pada event di bagian akhir hanya dipakai sebagai diagnostik awal, bukan sebagai label turnover final. Pipeline utama tetap mendeteksi turnover dari `ball_owning_team_id` pada tracking data.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MATCH_ID = 'J03WPY'
DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
tracking_path = DATA_DIR / f'{MATCH_ID}_tracking.csv'
events_path = DATA_DIR / f'{MATCH_ID}_events.csv'

tracking = pd.read_csv(tracking_path)
events = pd.read_csv(events_path)
tracking['timestamp'] = pd.to_timedelta(tracking['timestamp'], errors='coerce')
events['timestamp'] = pd.to_timedelta(events['timestamp'], errors='coerce')

print('Tracking:', tracking_path)
print('Events  :', events_path)
print('Tracking shape:', tracking.shape)
print('Events shape  :', events.shape)

Tracking: C:\VAEP-Track\data\sportec_idsse\J03WPY_tracking.csv
Events  : C:\VAEP-Track\data\sportec_idsse\J03WPY_events.csv
Tracking shape: (146211, 133)
Events shape  : (1572, 19)


In [2]:
player_ids = sorted({
    column[:-2]
    for column in tracking.columns
    if column.startswith('DFL-OBJ-') and column.endswith('_x')
})
alive_tracking = tracking[tracking['ball_state'].eq('alive')]

audit_summary = pd.DataFrame({
    'dataset': ['tracking', 'events'],
    'rows': [len(tracking), len(events)],
    'columns': [tracking.shape[1], events.shape[1]],
    'periods': [tracking['period_id'].nunique(), events['period_id'].nunique()],
    'timestamp_missing': [tracking['timestamp'].isna().sum(), events['timestamp'].isna().sum()],
    'memory_mb': [
        tracking.memory_usage(deep=True).sum() / 1024**2,
        events.memory_usage(deep=True).sum() / 1024**2,
    ],
})
display(audit_summary.round({'memory_mb': 2}))

print('Tracking player IDs:', len(player_ids))
print(
    'Alive frames with owning team:',
    f"{alive_tracking['ball_owning_team_id'].notna().mean():.2%}",
)

display(tracking[[
    'period_id', 'timestamp', 'frame_id', 'ball_state',
    'ball_owning_team_id', 'ball_x', 'ball_y', 'ball_z', 'ball_speed',
]].head())
display(events[[
    'event_id', 'event_type', 'period_id', 'timestamp', 'team_id',
    'player_id', 'success', 'coordinates_x', 'coordinates_y',
]].head())

display(events['event_type'].value_counts().rename('count').to_frame())
display(events['team_id'].value_counts(dropna=False).rename('count').to_frame())

,dataset,rows,columns,periods,timestamp_missing,memory_mb
0,tracking,146211,133,2,0,162.38
1,events,1572,19,2,0,0.87


Tracking player IDs: 31
Alive frames with owning team: 100.00%


,period_id,timestamp,frame_id,ball_state,ball_owning_team_id,ball_x,ball_y,ball_z,ball_speed
0,1,0 days 00:00:00,10000,alive,DFL-CLU-00000P,0.560952,0.500588,0.15,60.77
1,1,0 days 00:00:00.040000,10001,alive,DFL-CLU-00000P,0.566571,0.500294,0.18,59.15
2,1,0 days 00:00:00.080000,10002,alive,DFL-CLU-00000P,0.572381,0.500147,0.21,54.09
3,1,0 days 00:00:00.120000,10003,alive,DFL-CLU-00000P,0.578095,0.499559,0.25,53.68
4,1,0 days 00:00:00.160000,10004,alive,DFL-CLU-00000P,0.583429,0.499265,0.24,53.90


,event_id,event_type,period_id,timestamp,team_id,player_id,success,coordinates_x,coordinates_y
0,18237400000006,PASS,1,0 days 00:00:00,DFL-CLU-00000P,DFL-OBJ-002G5J,True,0.500000,0.500000
1,18237400000007,PASS,1,0 days 00:00:01.697000,DFL-CLU-00000P,DFL-OBJ-0028FW,True,0.718476,0.486765
2,18237400000008,GENERIC:TacklingGame,1,0 days 00:00:04.936000,NaN,NaN,NaN,0.335905,0.913382
3,18237400000009,PASS,1,0 days 00:00:06.465000,DFL-CLU-00000P,DFL-OBJ-002FXT,False,0.295810,0.923235
4,18237400000010,PASS,1,0 days 00:00:11.553000,DFL-CLU-000005,DFL-OBJ-0001HW,True,0.084000,0.680000


,count
event_type,
PASS,891
GENERIC:OtherBallAction,252
GENERIC:TacklingGame,208
BALL_OUT,70
GENERIC:Delete,60
SHOT,26
FOUL_COMMITTED,18
RECOVERY,12
CARD,9


,count
team_id,
DFL-CLU-00000P,785
DFL-CLU-000005,495
NaN,292


In [3]:
tracking_is_ordered = tracking.groupby('period_id')['timestamp'].apply(
    lambda values: values.is_monotonic_increasing
).all()
events_is_ordered = events.groupby('period_id')['timestamp'].apply(
    lambda values: values.is_monotonic_increasing
).all()
print('Tracking chronological in each period:', tracking_is_ordered)
print('Events chronological in source order  :', events_is_ordered)

team_events = (
    events.dropna(subset=['team_id'])
    .sort_values(['period_id', 'timestamp', 'event_id'])
    .reset_index(drop=True)
)
team_events['prev_team_id'] = team_events.groupby('period_id')['team_id'].shift(1)
diagnostic_changes = team_events[
    team_events['prev_team_id'].notna()
    & team_events['team_id'].ne(team_events['prev_team_id'])
].copy()

print('Diagnostic event-team changes:', len(diagnostic_changes))
display(diagnostic_changes[[
    'period_id', 'timestamp', 'event_type', 'prev_team_id',
    'team_id', 'player_id', 'success',
]].head(30))
print(
    'Catatan: hasil ini hanya sanity check event chronology; '
    'turnover final dihitung dari perubahan ball_owning_team_id pada tracking.'
)

Tracking chronological in each period: True
Events chronological in source order  : False
Diagnostic event-team changes: 405


,period_id,timestamp,event_type,prev_team_id,team_id,player_id,success
3,1,0 days 00:00:11.553000,PASS,DFL-CLU-00000P,DFL-CLU-000005,DFL-OBJ-0001HW,True
7,1,0 days 00:00:24.490000,GENERIC:OtherBallAction,DFL-CLU-000005,DFL-CLU-00000P,DFL-OBJ-002GMO,NaN
8,1,0 days 00:00:26.490000,GENERIC:OtherBallAction,DFL-CLU-00000P,DFL-CLU-000005,DFL-OBJ-002595,NaN
9,1,0 days 00:00:27.130000,GENERIC:OtherBallAction,DFL-CLU-000005,DFL-CLU-00000P,DFL-OBJ-0000F8,NaN
10,1,0 days 00:00:28.210000,GENERIC:OtherBallAction,DFL-CLU-00000P,DFL-CLU-000005,DFL-OBJ-0028BZ,NaN
11,1,0 days 00:00:28.890000,GENERIC:OtherBallAction,DFL-CLU-000005,DFL-CLU-00000P,DFL-OBJ-0000NZ,NaN
12,1,0 days 00:00:30.730000,PASS,DFL-CLU-00000P,DFL-CLU-000005,DFL-OBJ-0028T3,False
13,1,0 days 00:00:31.050000,GENERIC:OtherBallAction,DFL-CLU-000005,DFL-CLU-00000P,DFL-OBJ-J01H9X,NaN
14,1,0 days 00:00:38.129000,GENERIC:OtherBallAction,DFL-CLU-00000P,DFL-CLU-000005,DFL-OBJ-0000EJ,NaN
15,1,0 days 00:00:47.832000,PASS,DFL-CLU-000005,DFL-CLU-00000P,DFL-OBJ-0000NZ,True


Catatan: hasil ini hanya sanity check event chronology; turnover final dihitung dari perubahan ball_owning_team_id pada tracking.
